## Setup

In [7]:
from arcgis.gis import GIS
import arcpy
from arcgis.features import FeatureLayer
import pandas as pd
import json
import urllib.request
import urllib.parse
import os
import glob
import tempfile

In [18]:
gis = GIS('pro')
print(gis)
aprx = arcpy.mp.ArcGISProject("CURRENT")
map = aprx.activeMap
arcpy.env.addOutputsToMap = False
arcpy.env.overwriteOutput = True
project_path = r"C:\Users\micha\OneDrive\Documents\ArcGIS\Projects\California Watershed Delineation Sandbox"
working_gdb = os.path.join(project_path, r"California Watershed Delineation Sandbox.gdb")

GIS @ https://www.arcgis.com/ version:[2026, 2]


In [ ]:
# Getting all the TIF files
download_list = pd.read_csv(os.path.join(project_path, r"USGS DEMs\Data_scrubbed.csv"))

output_folder = os.path.join(project_path, r"USGS DEMs")
os.makedirs(output_folder, exist_ok=True)

for url in download_list['downloadURL2']:
    filename = os.path.join(output_folder, os.path.basename(url))
    urllib.request.urlretrieve(url, filename)
    print(f"Downloaded {filename}")

In [ ]:
# I ended up archiving most of the TIFs because it's way too huge to run all the data in California
tif_files = glob.glob(os.path.join(project_path, "USGS DEMs", "*.tif"))

# Now I'm combining the 5 files of interest into one raster dataset
arcpy.management.MosaicToNewRaster(
    input_rasters=tif_files,
    output_location=working_gdb,
    raster_dataset_name_with_extension="dem_mosaic",
    pixel_type="32_BIT_FLOAT",
    number_of_bands=1
)

In [ ]:
# Okay there are some odd lines on these raster boundaries, all at 0 elevation. Let's just remove those
# If any coastline happens to be registered as true 0.000, we'll lose maybe a few cells (inconsequent)
sea_level_masked = arcpy.sa.SetNull(
    in_conditional_raster=os.path.join(working_gdb, "dem_mosaic"),
    in_false_raster_or_constant=os.path.join(working_gdb, "dem_mosaic"),
    where_clause="VALUE = 0"
)
sea_level_masked.save(os.path.join(working_gdb, "dem_setnull"))

In [ ]:
# Let's reproject for meters
arcpy.management.ProjectRaster(
    in_raster=sea_level_masked,
    out_raster=os.path.join(working_gdb, "sea_level_projected"),
    out_coor_system=arcpy.SpatialReference(3310)  # California Albers, a standard CA projected system in meters
)
arcpy.management.Delete(os.path.join(working_gdb, "dem_setnull"))

In [ ]:
# Fill the DEM data
filled_dem = arcpy.sa.Fill(os.path.join(working_gdb, "sea_level_projected"))
filled_dem.save(os.path.join(working_gdb, "dem_clean"))
arcpy.management.Delete(os.path.join(working_gdb, "sea_level_projected"))

In [6]:
# Calculate flow direction
flow_dir = arcpy.sa.FlowDirection(os.path.join(working_gdb, "dem_clean"))
flow_dir.save(os.path.join(working_gdb, "flow_direction"))

In [7]:
# Calculation flow accumulation
flow_acc = arcpy.sa.FlowAccumulation(flow_dir)
flow_acc.save(os.path.join(working_gdb, "flow_accumulation"))

In [ ]:
# I was already writing Python so I made a feature class for a pour point here
arcpy.management.CreateFeatureclass(
    out_path=working_gdb,
    out_name="pour_point",
    geometry_type="POINT",
    spatial_reference=arcpy.Describe(os.path.join(working_gdb, "dem_clean")).spatialReference
)

In [43]:
# Next I'd like to make sure that the point I placed on the map is actually on a raster cell of high flow
snapped_point = arcpy.sa.SnapPourPoint(
    in_pour_point_data=os.path.join(working_gdb, r"pour_point_2"),
    in_accumulation_raster=os.path.join(working_gdb, r"flow_accumulation"),
    snap_distance=100 # search radius in map units (meters)
)
snapped_point.save(os.path.join(working_gdb, "pour_point_snapped"))

In [44]:
# Finally, we can create our watershed 
watershed_result = arcpy.sa.Watershed(os.path.join(working_gdb, r"flow_direction"), os.path.join(working_gdb, r"pour_point_snapped"))
watershed_result.save(os.path.join(working_gdb, r"Watershed_superfund_downstream"))

In [ ]:
# Create a clean watershed polygon to later run intersects with
arcpy.conversion.RasterToPolygon(
    os.path.join(working_gdb, "Watershed_superfund_downstream"),
    os.path.join(working_gdb, "watershed_polygon_superfund")
)
arcpy.management.Dissolve(
    os.path.join(working_gdb, "watershed_polygon_superfund"),
    os.path.join(working_gdb, "watershed_superfund_dissolved")
)

In [46]:
# Filter out streams with 5,000 contribution cells or less
streams = arcpy.sa.Con(os.path.join(working_gdb, "flow_accumulation"), 1, 0, "VALUE > 5000")
streams.save(os.path.join(working_gdb, "streams"))

# Clip the streams to the region of interest
streams_clipped = arcpy.sa.ExtractByMask(
    os.path.join(working_gdb, "streams"),
    os.path.join(working_gdb, "watershed_superfund_dissolved")
)
# Set null for future analysis
streams_true = arcpy.sa.SetNull(
    streams_clipped,
    streams_clipped,
    "VALUE = 0"
)
streams_true.save(os.path.join(working_gdb, "streams_clipped"))
arcpy.management.delete(os.path.join(working_gdb, "streams"))

In [48]:
# Clip the flow direction to the region of interest
flow_dir_clipped = arcpy.sa.ExtractByMask(
    os.path.join(working_gdb, "flow_direction"),
    os.path.join(working_gdb, "watershed_polygon_superfund")
)
flow_dir_clipped.save(os.path.join(working_gdb, "flow_dir_clipped"))
arcpy.management.delete(os.path.join(working_gdb, "flow_direction"))

In [49]:
# Create a stream link
streams = os.path.join(working_gdb, "streams_clipped")
flow_dir = os.path.join(working_gdb, "flow_dir_clipped")
stream_link = arcpy.sa.StreamLink(streams, flow_dir)
stream_link.save(os.path.join(working_gdb, "stream_link"))

In [ ]:
# Calculate the stats for the stream link
arcpy.sa.ZonalStatisticsAsTable(
    in_zone_data=os.path.join(working_gdb, "stream_link"),
    zone_field="VALUE",
    in_value_raster=os.path.join(working_gdb, "flow_accumulation"),
    out_table=os.path.join(working_gdb, "segment_flow_stats"),
    statistics_type="MAXIMUM"
)

In [ ]:
# Create a feature class from the stream link
arcpy.sa.StreamToFeature(
    in_stream_raster=os.path.join(working_gdb, "stream_link"),
    in_flow_direction_raster=os.path.join(working_gdb, "flow_dir_clipped"),
    out_polyline_features=os.path.join(working_gdb, "streams_superfund_weighted"),
    simplify="SIMPLIFY"
)

In [ ]:
# Finally, add the zonal stats to the stream link feature class so I can symbolize based on flow accumulation
arcpy.management.JoinField(
    in_data=os.path.join(working_gdb, "streams_superfund_weighted"),
    in_field="grid_code",
    join_table=os.path.join(working_gdb, "segment_flow_stats"),
    join_field="Value",
    fields=["MAX"]
)

In [ ]:
arcpy.management.CreateFeatureclass(
    out_path=working_gdb,
    out_name="site_crossings",
    geometry_type="POINT",
    spatial_reference=arcpy.Describe(os.path.join(working_gdb, "flow_direction")).spatialReference
)
arcpy.management.AddField(os.path.join(working_gdb, "site_crossings"), "direction", "TEXT")

In [ ]:
# Filter to your 3 real Superfund sites
arcpy.analysis.Select(
    in_features=os.path.join(working_gdb, "superfund_boundary"),
    out_feature_class=os.path.join(working_gdb, "superfund_sites_filtered"),
    where_clause="EPA_ID IN ('CAD980894893', 'CAD980894901', 'CAD980894976')"
)

# Boolean mask: is this cell inside the filtered site boundary?
# ExtractByMask clips the DEM to the site polygon; cells outside become NoData.
# IsNull flags those NoData cells (True where outside), then Con flips that
# into a clean 1 = inside / 0 = outside raster.
site_bool = arcpy.sa.Con(arcpy.sa.IsNull(arcpy.sa.ExtractByMask(
    os.path.join(working_gdb, "dem_clean"), os.path.join(working_gdb, "superfund_sites_filtered")
)), 0, 1)
site_bool.save(os.path.join(working_gdb, "site_bool"))

flow_dir = os.path.join(working_gdb, "flow_dir_clipped")
cellsize = arcpy.Describe(flow_dir).meanCellHeight

direction_offsets = {
    1: (0, 1), 2: (1, 1), 4: (1, 0), 8: (1, -1),
    16: (0, -1), 32: (-1, -1), 64: (-1, 0), 128: (-1, 1)
}

# Work in a scratch TIFF folder, not the gdb — the repeated save/delete/rename
# cycle below is unreliable against file geodatabase raster storage.
scratch_folder = tempfile.mkdtemp()

downstream_in_site_path = os.path.join(scratch_folder, "downstr_in_site.tif")
downstream_in_site_tmp_path = os.path.join(scratch_folder, "downstr_in_site_tmp.tif")

# For each of the 8 directions: shift the "inside site" mask by that direction's
# offset, then check where flow_dir actually matches that direction. Wherever it
# does, this cell's downstream neighbor is (per this direction) inside the site.
# Summing across all 8 directions gives one raster where every cell reflects
# whether ITS OWN flow direction — whatever it happens to be — leads inside.
# Note: x-offset (dc) is negated; Shift's x_value moves raster content east/west,
# which is the opposite sense needed to sample "the neighbor at column+dc."
for i, (code, (dr, dc)) in enumerate(direction_offsets.items()):
    shifted_path = os.path.join(scratch_folder, f"shift_{code}.tif")
    arcpy.management.Shift(site_bool, shifted_path, -dc * cellsize, dr * cellsize)
    shifted = arcpy.Raster(shifted_path)

    matched = arcpy.sa.Con(flow_dir, shifted, 0, f"VALUE = {code}")

    if i == 0:
        matched.save(downstream_in_site_path)
    else:
        combined = arcpy.sa.CellStatistics([arcpy.Raster(downstream_in_site_path), matched], "SUM")
        combined.save(downstream_in_site_tmp_path)
        arcpy.management.Delete(downstream_in_site_path)
        arcpy.management.Rename(downstream_in_site_tmp_path, downstream_in_site_path)

    arcpy.management.Delete(shifted_path)
    print(f"Processed direction {code}")

# Bring the final result into your actual gdb, once, at the end
arcpy.management.CopyRaster(downstream_in_site_path, os.path.join(working_gdb, "downstream_in_site_final"))

In [ ]:
streams = os.path.join(working_gdb, "streams_clipped")
site_bool_ras = os.path.join(working_gdb, "site_bool")
downstream = os.path.join(working_gdb, "downstream_in_site_final")

# ID outs by filtering my streams where they are both inside the site boundary, and where their nearest downstream cell is outside the site boundary
out_points = arcpy.sa.Con(streams, arcpy.sa.Con((arcpy.Raster(site_bool_ras) == 1) & (arcpy.Raster(downstream) == 0), 1))
out_points.save(os.path.join(working_gdb, "out_points"))

# ID ins by filtering my streams where they are both outside the site boundary, and where their nearest downstream cell is inside the site boundary
in_points = arcpy.sa.Con(streams, arcpy.sa.Con((arcpy.Raster(site_bool_ras) == 0) & (arcpy.Raster(downstream) == 1), 1))
in_points.save(os.path.join(working_gdb, "in_points"))

# Generate point layers and add coordinated
arcpy.conversion.RasterToPoint(out_points, os.path.join(working_gdb, "out_points_pts"))
arcpy.conversion.RasterToPoint(in_points, os.path.join(working_gdb, "in_points_pts"))
arcpy.management.AddXY(os.path.join(working_gdb, "out_points_pts"))
arcpy.management.AddXY(os.path.join(working_gdb, "in_points_pts"))

In [ ]:
# Sum accumulation over a set of points
def sum_accum_at(points_fc, accum_raster):
    total = 0
    with arcpy.da.SearchCursor(points_fc, ["SHAPE@X", "SHAPE@Y"]) as cursor:
        for row in cursor:
            x, y = row
            val = arcpy.management.GetCellValue(accum_raster, f"{x} {y}")
            total += float(val[0])
    return total

# Run the function on the list of points I just generated
outs = sum_accum_at(os.path.join(working_gdb, "out_points_pts"), os.path.join(working_gdb, "flow_accumulation"))
ins = sum_accum_at(os.path.join(working_gdb, "in_points_pts"), os.path.join(working_gdb, "flow_accumulation"))


In [30]:
# Determine superfund site contribution to the pour point
site_contribution = outs - ins
fraction = site_contribution / 14695295 # Python was getting finicky with the datasets here so I just took the number from the map
print(f"Site contributes {site_contribution} accumulation units — {fraction:.1%} of total flow")

Site contributes 386787.0 accumulation units — 2.6% of total flow
